In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import os,sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from matplotlib.text import Text
import seaborn as sns

##import umap.plot
%matplotlib inline

# set root directory
sys.path.insert(0, str(pathlib.Path.cwd()))
import az_utils.utils as utils

utils.apply_defaults(set_cwd=True)


# get some stuff
label_order_controls, cell_lines, color_dict_cells, color_dict_controls = utils.define_colors()
from az_utils.replicate import corr_between_non_replicates, corr_between_replicates, corr_between_non_replicates, percent_score


In [ ]:
BaseDir = "./results-medmean/"

OutputDir = str(figdir("SupplFig4")) + "/"
" + EXP4_TAGS + "
if not os.path.exists(OutputDir):
    os.makedirs(OutputDir)

figformat = 'pdf'
dpi = 300

In [ ]:
# Import data
selected_df = pd.read_parquet(profiles("exp4_objective", "selected_data_HT29.parquet"))

selected_df["Metadata_name"] = selected_df["Metadata_cmpdname"].str[:4]

selected_df.shape


In [ ]:
# Import metadata
# `metadata` the helper is shadowed by the dataframe below, so resolve first.
_meta_csv = metadata('spher-colo6-az-version5.csv', 'exp4_objective')
metadata = pd.read_csv(_meta_csv, sep="," , usecols=['barcode', 'well_id', 'cmpdname', 'cmpd_conc', 'pert_type'])
metadata.drop_duplicates(inplace=True)

display(metadata.shape)

# Prepend Metadata to all of them 
rename_dict = {'barcode':'Metadata_Barcode', 'well_id':'Metadata_Well', 'cmpdname':'Metadata_cmpdname', 'cmpd_conc':'Metadata_cmpd_conc', 'pert_type':'Metadata_pert_type'}
metadata = metadata.rename(columns= rename_dict)

# Make metadata_name the short version
metadata["Metadata_name"] = metadata["Metadata_cmpdname"].str[:4]

# Make metadata_name the short version
metadata["Metadata_cmpd_conc"] = metadata["Metadata_cmpd_conc"].astype(str)

# ## Attach it to selected_df
# selection2 = pd.merge(metadata, selected_df, how='outer', on=['Metadata_Barcode', 'Metadata_Well', 'Metadata_cmpdname', 'Metadata_name','Metadata_cmpd_conc', 'Metadata_pert_type' ])

# ## Add a new metadata for plotting the missing wells 
# selection2['missing'] = selection2['Metadata_PlateWell'].isnull()


# display(selection2.shape)

In [ ]:
# Run the calculations to get spearman correlations
grouping = ['Metadata_cmpd_conc', 'Metadata_name', 'Metadata_cell_line', 'Metadata_Barcode']

compounds_df = utils.get_featuredata(selected_df.query("Metadata_pert_type == 'pos_con'").set_index(grouping))

corrs_df = pd.concat([corr_between_replicates(compounds_df, grouping, method='spearman'), 
                      corr_between_non_replicates(compounds_df, grouping, method='spearman')], 
                      axis=1).reset_index()

# corrs_df.to_csv("{}/replicating_correlations.csv".format(BaseDir))

In [ ]:
# # Add a column indicating whether the correlation could not be made

# corrs_df['missing'] = False
# for idx, row in corrs_df.iterrows():
#     if pd.isna(row['corr']):
#         corrs_df.at[idx, 'missing'] = True
#         corrs_df.at[idx, 'corr'] = 0


In [ ]:
# Calculate percent replicating per plateXconcentration
#  
records = []

for (conc, plate), group in corrs_df.groupby(['Metadata_cmpd_conc', 'Metadata_Barcode']):
    
    pr, *_ = percent_score(group['null_corr'],group['corr'])
    records.append({
        'Metadata_cmpd_conc': conc,
        'Metadata_Barcode': plate,
        'percent_replicating': pr
    })

repl_df = pd.DataFrame.from_records(records)

# repl_df.to_csv("{}/percent_replicating.csv".format(BaseDir))

#### Summary plot

In [ ]:
# Plot the data in a swarmplot

# Create scatterplots with random jitter for each concentration
fig, ax = plt.subplots(figsize=(7, 5))
sns.swarmplot(data = repl_df,  
              x='Metadata_cmpd_conc', 
              y='percent_replicating', 
              hue='Metadata_Barcode', 
              ax=ax, 
              alpha=0.9, 
              s=8, 
            #   hue_order=cell_lines,
            #   palette=color_dict_cells,
              )

ax.set_xlabel('Concentration (µM)', fontsize=16)
ax.set_ylabel('Percent Replicating', fontsize=16)


plt.title('Reproducibility across batches', fontsize=16)

plt.legend(title='Barcode', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.xticks(rotation=45)
# ax.set_ylim(0, 100)
plt.show()

# fig.savefig(
# "{}/Reproducibility_across_celllines.{}".format(OutputDir, "pdf"), dpi=dpi, bbox_inches="tight", transparent=True
# )

In [ ]:
# Group the metadata by barcode, name and concentration == perturbation

metadata_short = metadata[['Metadata_Barcode', 'Metadata_name', 'Metadata_cmpd_conc']].drop_duplicates()



In [ ]:
corrs2_df = pd.merge(left=corrs_df, right=metadata_short, on= ['Metadata_Barcode', 
                                             'Metadata_name',
                                             'Metadata_cmpd_conc'], how='outer')

# corrs_df = corrs2_df

In [ ]:
# Add a column indicating whether the correlation could not be made

corrs2_df['missing'] = False
for idx, row in corrs2_df.iterrows():
    if pd.isna(row['null_corr']):
        corrs2_df.at[idx, 'missing'] = True
        corrs2_df.at[idx, 'corr'] = 0

In [ ]:

corrs2_df.drop(corrs2_df[(corrs2_df['Metadata_name'] == 'DMSO') | (corrs2_df['Metadata_name'] == 'wate')].index, inplace=True)

### Plot the pairwise correlations for each compound separately

In [ ]:
## Scatterplot of correlations between replicates for compounds

# One output per acquisition. NOTE: the published Suppl 4h/4i "air" panel is the two
# non-WI acquisitions COMBINED (see PORT_TRIAGE); this notebook emits them separately.
# The port injected this block into 3_PCA_objective.ipynb but not here, so PLATE_TAG
# was used below without ever being defined.
PLATE_TAG = {
    'CellPainting_20241220clearedspheroidsBOMI_20241220_151510': 'bomi',
    'CellPainting_20250127Cellpaintcleared3D_20250127_171120':   'cleared3d',
    'CellPainting_CellPaint3DBomi_WI_for_Jordi_20250203_155142': 'wi',
}

plates = corrs2_df.Metadata_Barcode.unique()

for i, plate in enumerate(plates):
    # ax = axes[i]
    fig = plt.figure(figsize=(8,5))
    plt.title(plate, fontsize=14,y=1.05 )

    # Collect the data
    df = corrs2_df.query('Metadata_Barcode == @plate').copy()
    df["Metadata_conc_num"] = round(pd.to_numeric(df["Metadata_cmpd_conc"], errors="raise")*1000,2)
    df = df.sort_values("Metadata_conc_num", ascending=True)
    
    # Create scatterplots with random jitter for each concentration
    ax = sns.swarmplot(x='Metadata_conc_num', y='corr', data=df, 
                #   ax=ax, 
                  alpha=0.9, s=6, hue="Metadata_name",
                  palette=color_dict_controls,
                  )

    # Add a line at the 95th percentile
    perc_95 = np.nanpercentile(df['null_corr'], 95)
    ax.axhline(perc_95, ls='--', color='k', label='95th percentile')

    ax.set_xlabel("concentration [µm]", fontsize=14)
    ax.set_ylabel('Spearman correlation',fontsize=14)
    ax.set_ylim([-0.25, 1])
    plt.xticks(rotation=45)
    plt.legend(title='compound', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.tight_layout()
    plt.show()

    save_panel(fig, f'SupplFig4i_{PLATE_TAG[plate]}',
               data=corrs2_df[corrs2_df.Metadata_Barcode == plate],
               caption=f'Replicate Spearman correlation vs concentration, {plate}',
               notebook='analysis/3_SupplFigure4/3_Fig_TechnicalReplicates.ipynb')

